# no-grad-context-mgr-update — faded example 2: complete NoGrad.__exit__

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `no-grad-context-mgr-update`. Running the beacon reports progress on the `Backprop: no_grad ctx-mgr update` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: no_grad ctx-mgr update` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`no-grad-context-mgr-update`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "no-grad-context-mgr-update"
DD_SUBTOPIC = "Backprop: no_grad ctx-mgr update"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

On exit, `NoGrad` must restore the PREVIOUS value it saved, not a hard-coded `True`. Restoring the saved value is what lets nested blocks behave correctly.

## Faded exercise 2

Complete `NoGrad.__exit__` so it restores the saved previous flag value. Fill in the restore line.

**Fill in:** the line restoring grad_tracking_enabled from self._prev

In [ ]:
grad_tracking_enabled = True

class NoGrad:
    def __enter__(self):
        global grad_tracking_enabled
        self._prev = grad_tracking_enabled
        grad_tracking_enabled = False
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        global grad_tracking_enabled
        raise NotImplementedError()  # TODO: the line restoring grad_tracking_enabled from self._prev

with NoGrad():
    pass
print(grad_tracking_enabled)


def _test():
    global grad_tracking_enabled
    grad_tracking_enabled = True
    with NoGrad():
        pass
    assert grad_tracking_enabled is True
    # exception path must also restore
    try:
        with NoGrad():
            raise RuntimeError('x')
    except RuntimeError:
        pass
    assert grad_tracking_enabled is True


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
grad_tracking_enabled = True

class NoGrad:
    def __enter__(self):
        global grad_tracking_enabled
        self._prev = grad_tracking_enabled
        grad_tracking_enabled = False
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        global grad_tracking_enabled
        grad_tracking_enabled = self._prev

with NoGrad():
    pass
print(grad_tracking_enabled)
```
</details>